In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri
from mpi4py import MPI
from dolfinx import mesh, fem
from dolfinx.fem.petsc import LinearProblem
import ufl


In [ ]:
# ── 1. MESH ──────────────────────────────────────────────────────────────────
# 16x16 grid to match your Neural Operator parametrisation
Nx, Ny = 64, 64
domain = mesh.create_unit_square(MPI.COMM_WORLD, Nx, Ny)

In [ ]:
# ── 2. PERMEABILITY FIELD a(x) ───────────────────────────────────────────────
# Piecewise-constant per cell (DG0): one value per triangle
# With a 16x16 quad mesh you get 16*16*2 = 512 triangles
V_k = fem.functionspace(domain, ("DG", 0))
a_func = fem.Function(V_k)

# u_proposal = sample_GP()
# theta_proposal = smooth_threshold(u_proposal)  # threshold to get binary field

# # Replace this with your actual a(x) sample:
# pixel_indices = np.repeat(theta_proposal.ravel(), 2)  # shape (512,)
# a_func.x.array[:] = pixel_indices

from dolfinx import geometry

# Get midpoint of each cell
tdim = domain.topology.dim
num_cells = domain.topology.index_map(tdim).size_local
midpoints = mesh.compute_midpoints(domain, tdim, np.arange(num_cells))

# Map each midpoint (x,y) → pixel value
ix = np.clip((midpoints[:, 0] * Nx).astype(int), 0, Nx - 1)
iy = np.clip((midpoints[:, 1] * Ny).astype(int), 0, Ny - 1)

theta_proposal = dataset.test_dbs[16].x[0, 0]
print(theta_proposal.shape)
a_func.x.array[:] = np.where(theta_proposal[iy, ix] > 0.5, 1.0, 1e-3)

torch.Size([16, 16])


In [ ]:
# ── 3. FUNCTION SPACE & BCs ──────────────────────────────────────────────────
V = fem.functionspace(domain, ("Lagrange", 1))

# u = 0 on ALL of ∂Ω (homogeneous Dirichlet)
fdim = domain.topology.dim - 1
all_boundary_facets = mesh.locate_entities_boundary(
    domain, fdim, lambda x: np.full(x.shape[1], True, dtype=bool)
)
bc = fem.dirichletbc(
    fem.Constant(domain, 0.0),
    fem.locate_dofs_topological(V, fdim, all_boundary_facets),
    V
)

In [ ]:
# ── 4. VARIATIONAL FORM ──────────────────────────────────────────────────────
# PDE: -∇·(a(x)∇u) = 1  →  ∫ a(x) ∇u·∇v dx = ∫ v dx
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

a_form = a_func * ufl.dot(ufl.grad(u), ufl.grad(v)) * ufl.dx
L_form = v * ufl.dx     # RHS = 1 everywhere

In [ ]:
# ── 5. SOLVE ─────────────────────────────────────────────────────────────────
problem = LinearProblem(a_form, L_form, bcs=[bc], petsc_options_prefix="darcy", petsc_options={
    "ksp_type": "cg",
    "pc_type": "hypre",
    "pc_hypre_type": "boomeramg"
})
uh = problem.solve()

In [ ]:
# ── 6. EXTRACT SOLUTION ON 16x16 GRID ───────────────────────────────────────
# DOF coordinates for P1 space → (Nx+1)*(Ny+1) = 17*17 = 289 points
coords = V.tabulate_dof_coordinates()[:, :2]
u_vals = uh.x.array.real

# Interpolate onto a regular 16x16 grid (matching your NO output grid)
from scipy.interpolate import griddata
grid_x, grid_y = np.meshgrid(
    np.linspace(0, 1, Nx), np.linspace(0, 1, Ny)
)
u_grid = griddata(coords, u_vals, (grid_x, grid_y), method="linear")